In [143]:
import os
from re import search
from unittest import result

import certifi
import dotenv


from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_tavily import TavilySearch
from langsmith import Client as LangSmithClient
from langchain.tools import tool
import requests




In [144]:
from langchain_classic.agents import create_react_agent, AgentExecutor


In [145]:
from dotenv import load_dotenv

# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [146]:
search_tool = TavilySearch(max_results=2)

In [147]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [148]:
print(get_weather_data.invoke("hadera"))

City: hadera
Temperature: 30°C
Weather: Sunny
Humidity: 55%


In [149]:
result = search_tool.invoke("Give me the latest news on AI")
result

{'query': 'Give me the latest news on AI',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.artificialintelligence-news.com',
   'title': 'AI News | Latest News | Insights Powering AI-Driven Business Growth',
   'content': 'July 30, 2026\n\n### Google’s Gemini 3.6 Flash targets enterprise agent token costs\n\nAI in Action\n\nJuly 21, 2026\n\n### HP accelerates enterprise workflows with OpenAI Frontier\n\nWorld of Work\n\nJune 29, 2026\n\n#### Deep Learning\n\n### Aviva deploys AI to stop £230M in sophisticated insurance fraud\n\nAI in Action\n\nJune 8, 2026\n\n### China’s AI just mapped its entire renewable energy grid. Here’s why the rest of the world should pay attention\n\nEnvironment & Sustainability\n\nMay 22, 2026 [...] Artificial Intelligence\n\nAugust 5, 2026\n\n# Alibaba, DeepSeek push China’s AI model race towards lower costs\n\nGovernance, Regulation & Policy\n\nAugust 4, 2026\n\n# Red Hat, NVIDIA, IBM back project turning AI po

In [150]:
# ==========================================
# LLM
# ==========================================

# gemini-flash-lite-latest has free-tier quota available on this key
# (gemini-3.5-flash caps at 20 req/day; gemini-2.0-flash-lite/flash
# had zero free quota on this key/region). A ReAct agent burns one
# request per Thought/Action step, so pace requests client-side too.
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.5,  # one request every 2s, stays clear of per-minute caps
    check_every_n_seconds=0.1,
    max_bucket_size=1,
)

llm = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0,
    google_api_key=GEMINI_API_KEY,
    max_retries=3,
    rate_limiter=rate_limiter,
)
print(llm)

metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-google-genai': '4.3.2'}} rate_limiter=<langchain_core.rate_limiters.InMemoryRateLimiter object at 0x112d24390> output_version=None profile={'name': 'Gemini Flash-Lite Latest', 'release_date': '2026-05-07', 'last_updated': '2026-05-07', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True} google_api_key=SecretStr('**********') location=None model='gemini-flash-lite-latest' temperature=0.0 max_retries=3 client=<google.genai.client.Client object at 0x114361450> default_metadata=() model_kwarg

In [151]:
response = llm.invoke("Tell me a jock about AI")
response.content

[{'type': 'text',
  'text': "Why do artificial intelligence algorithms make terrible comedians?\n\nBecause they always rely on **machine learning**, but they haven't figured out the **human timing**!",
  'extras': {'signature': 'EjQKMgERTTIPwYMNKWVx9idpd3rDl9GmKB+KiZhrGszQJ77cLdDjVkVe0VaOj3UZGm0u5iws'}}]

In [152]:
# ==========================================
# PROMPT
# ==========================================

prompt = LangSmithClient().pull_prompt(
    "hwchase17/react", dangerously_pull_public_prompt=True
)

In [153]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [154]:
#======
# TOOLS
#======
tools = [search_tool, get_weather_data]

In [155]:
#=======
# CREATE AGENT
# ======

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)


In [162]:
#=====
# EXECUTOR
#======
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=5,
    max_execution_time=20,
    early_stopping_method="force",
    handle_parsing_errors=True,
)

In [170]:
#========
# RUN
#========

response = agent_executor.invoke({
        "input": (
            "Which is better for tourists, Rome or Paris?"
        )
    })



> Entering new AgentExecutor chain...
Question: Which is better for tourists, Rome or Paris?
Thought: I should search for travel comparisons between Rome and Paris to provide a comprehensive answer covering what each city offers tourists.
Action: tavily_search
Action Input: Rome vs Paris travel comparison which is better{'query': 'Rome vs Paris travel comparison which is better', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://embracesomeplace.com/paris-vs-rome', 'title': 'Paris vs Rome (If You Can Only Pick One)', 'content': 'Rome is cheaper, the food is more consistently amazing (and harder to mess up as a tourist), the city is more compact and walkable, the day trips are better, and the whole atmosphere is more forgiving. You can show up in jeans and sneakers and feel perfectly at home. The learning curve is gentler. The vibe is warmer. And eating your way through Trastevere on your first night will make you fall in love with travel in a way

In [171]:
print(response["output"])

Deciding whether Rome or Paris is better for tourists depends entirely on your personal interests, budget, and preferred travel style, as both are world-class destinations offering distinct experiences:

* **Choose Rome if you prefer:**
  * **Ancient History and Archaeology:** Rome is an open-air museum filled with iconic ancient landmarks like the Colosseum, the Roman Forum, the Pantheon, and Vatican City.
  * **Walkability and Compactness:** Most major historic sights are clustered closely together, making it easier to explore on foot without constantly relying on public transit.
  * **A More Relaxed Vibe:** Rome generally feels warmer, more casual, and slightly more budget-friendly. 
  * **Consistently Great Food:** Italian cuisine (pasta, pizza, gelato) is widely accessible, affordable, and universally loved.

* **Choose Paris if you prefer:**
  * **Art Museums and Grand Architecture:** Paris is renowned for world-class art institutions like the Louvre and the Musée d'Orsay, alongs